# Example 1.4.1 / 1: The sum Operator

Companion notebook for **1.4.1 Linear Expressions and sum** in the AMPL lab Documentation, section "The `sum` Operator". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## The `sum` Operator

`sum {indexing} expression` adds the expression over every member of the indexing set:

- `sum {p in PROD} profit[p] * Make[p]`: over one set
- `sum {i in ORIG, j in DEST} cost[i,j] * Trans[i,j]`: over all pairs of two sets
- `sum {(i,j) in LINKS} cost[i,j] * Trans[i,j]`: over a set of pairs

`sum` binds more tightly than `+` and `-`: it takes only the term right after it. To add a
longer expression, put it in parentheses. AMPL's `expand` command shows the difference: it
prints each constraint in the form AMPL sends to the solver. `ampl.get_output("expand ...;")` runs
AMPL's command and returns what it prints as a string. Here the model needs no data, so there is no `data.py`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set I = 1..3;      # three items
var x {I} >= 0;    # one amount per item
var y >= 0;        # a single shared amount
subject to Once:  sum {i in I} x[i] + y <= 5;      # y added once
subject to Thrice: sum {i in I} (x[i] + y) <= 5;   # y added for every i

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
print(ampl.get_output("expand Once, Thrice;"))

subject to Once:
	x[1] + x[2] + x[3] + y <= 5;

subject to Thrice:
	x[1] + x[2] + x[3] + 3*y <= 5;




### Pure AMPL: model.run

Pure AMPL:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
expand Once, Thrice;

subject to Once:
	x[1] + x[2] + x[3] + y <= 5;

subject to Thrice:
	x[1] + x[2] + x[3] + 3*y <= 5;

